# 04 — Tide Gauge Validation

Compare diffusion-model SSH predictions against NOAA tide gauge observations.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import LEAD_WEEKS, TIDE_GAUGES
from src.model.dataset import SSHDiffusionDataset
from src.model.sample  import sample_ensemble
from src.eof.reconstruct import reconstruct_field
from src.data.gauge_loader import load_gauges
from src.evaluation.gauge_comparison import interp_ssh_to_gauges
from src.evaluation.metrics import pearson_r, rmse, pairwise_corr_matrix, pattern_correlation
from src.evaluation.visualize import plot_skill_map, plot_correlation_matrices, plot_lead_time_skill

In [ ]:
# Generate validation forecasts for each (t, lead) pair
val_ds = SSHDiffusionDataset('val')

predictions = []  # one row per pair
for idx, (t, lead) in enumerate(val_ds.pairs):
    cond, target = val_ds[idx]
    ens_z = sample_ensemble(cond.numpy(), n_members=50)
    ens_mean = val_ds.denormalize_ssh(ens_z.mean(axis=0))
    predictions.append({
        't': t, 'lead': lead,
        'time_forecast': val_ds.time[t + lead],
        'pcs_pred': ens_mean,
        'pcs_obs':  val_ds.denormalize_ssh(target.numpy()),
    })
pred_df = pd.DataFrame(predictions)
pred_df.head()

In [ ]:
# Reconstruct SSH fields per lead and interpolate to gauges
skill_by_lead = {}
obs = load_gauges()

for lead in LEAD_WEEKS:
    sub = pred_df[pred_df.lead == lead].sort_values('time_forecast')
    pcs_pred = np.stack(sub.pcs_pred.values)
    fields = reconstruct_field(pcs_pred, 'ssh')
    model_gauges = interp_ssh_to_gauges(fields)
    model_gauges.index = pd.to_datetime(sub.time_forecast.values)
    model_gauges_monthly = model_gauges.resample('MS').mean()

    skill = {}
    for name in model_gauges.columns:
        if name not in obs.columns: continue
        aligned = pd.concat([model_gauges_monthly[name], obs[name]], axis=1, join='inner')
        skill[name] = pearson_r(aligned.iloc[:,1].values, aligned.iloc[:,0].values)
    skill_by_lead[lead] = skill

In [ ]:
# Skill vs lead time
skill_table = pd.DataFrame(skill_by_lead).T
skill_table.index.name = 'lead_weeks'
plot_lead_time_skill(skill_table)
skill_table

In [ ]:
# Station-pairwise correlation matrices (lead = 4 weeks, say)
lead = 4
sub = pred_df[pred_df.lead == lead].sort_values('time_forecast')
pcs_pred = np.stack(sub.pcs_pred.values)
fields = reconstruct_field(pcs_pred, 'ssh')
model_gauges = interp_ssh_to_gauges(fields)
model_gauges.index = pd.to_datetime(sub.time_forecast.values)

obs_mat  = pairwise_corr_matrix(obs)
pred_mat = pairwise_corr_matrix(model_gauges)

plot_correlation_matrices(obs_mat, pred_mat)
print(f'Pattern correlation: {pattern_correlation(obs_mat, pred_mat):.3f}')